In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

# racine projet
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent

PROJECT_ROOT = p
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
os.makedirs(DATA_PROCESSED, exist_ok=True)

print("DATA_RAW:", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)

DATA_RAW: c:\Users\azerty\Desktop\Projet2 - Copie\data\raw
DATA_PROCESSED: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed


In [2]:
akas_cols = ["titleId","ordering","title","region","types"]
akas = pd.read_csv(
    DATA_RAW / "title.akas.tsv.gz",
    sep="\t", compression="gzip",
    usecols=akas_cols, na_values="\\N",
    low_memory=True
)

akas_fr = akas[akas["region"] == "FR"].copy()
akas_fr["has_imdbDisplay"] = akas_fr["types"].fillna("").str.contains("imdbDisplay")

akas_fr = akas_fr.sort_values(["titleId","has_imdbDisplay","ordering"], ascending=[True, False, True])

akas_fr_best = (
    akas_fr.drop_duplicates("titleId", keep="first")
          .rename(columns={"titleId":"tconst", "title":"title_fr"})
          [["tconst","title_fr"]]
)

fr_ids = set(akas_fr_best["tconst"])
print("FR ids:", len(fr_ids))

FR ids: 5322015


In [3]:
basics_cols = ["tconst","titleType","primaryTitle","originalTitle","isAdult","startYear","runtimeMinutes","genres"]

chunks = []
for chunk in pd.read_csv(
    DATA_RAW / "title.basics.tsv.gz",
    sep="\t", compression="gzip",
    usecols=basics_cols, na_values="\\N",
    chunksize=300_000,  # safe
    low_memory=True
):
    chunk = chunk[(chunk["titleType"] == "movie") & (chunk["isAdult"] == 0)]
    chunk = chunk[chunk["tconst"].isin(fr_ids)]
    chunks.append(chunk)

basics = pd.concat(chunks, ignore_index=True)

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")
basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")

print("basics:", basics.shape)

C:\Users\azerty\AppData\Local\Temp\ipykernel_14248\1956492082.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\azerty\AppData\Local\Temp\ipykernel_14248\1956492082.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\azerty\AppData\Local\Temp\ipykernel_14248\1956492082.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\azerty\AppData\Local\Temp\ipykernel_14248\1956492082.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\azerty\AppData\Local\Temp\ipykernel_14248\1956492082.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
C:\Users\azerty\AppD

basics: (91143, 8)


In [4]:
ratings_cols = ["tconst","averageRating","numVotes"]
ratings = pd.read_csv(
    DATA_RAW / "title.ratings.tsv.gz",
    sep="\t", compression="gzip",
    usecols=ratings_cols, na_values="\\N",
    low_memory=True
)

ratings["averageRating"] = pd.to_numeric(ratings["averageRating"], errors="coerce")
ratings["numVotes"] = pd.to_numeric(ratings["numVotes"], errors="coerce")

imdb = basics.merge(ratings, on="tconst", how="inner")
imdb = imdb.merge(akas_fr_best, on="tconst", how="left")

# titre français obligatoire
imdb = imdb.dropna(subset=["title_fr"]).copy()

imdb["decade"] = (imdb["startYear"] // 10) * 10

print("imdb:", imdb.shape)

imdb: (73766, 12)


In [5]:
# enlever les valeurs manquantes nécessaires
imdb = imdb.dropna(subset=["runtimeMinutes", "averageRating", "numVotes"]).copy()

# filtres note + votes
imdb = imdb[(imdb["averageRating"] >= 6.0)].copy()

# filtre durée :
# - 60 à 120 minutes
# - jusqu'à 180 minutes SI note >= 8
cond_60_120 = imdb["runtimeMinutes"].between(60, 120)
cond_120_200_si_8 = (imdb["averageRating"] >= 8.0) & (imdb["runtimeMinutes"].between(60, 210))

imdb = imdb[cond_60_120 | cond_120_200_si_8].copy()

print("imdb after filters (rating/votes/runtime):", imdb.shape)

imdb after filters (rating/votes/runtime): (37167, 12)


In [6]:
imdb_small_path = DATA_PROCESSED / "imdb_small_fr.csv"
imdb.to_csv(imdb_small_path, index=False)
print("Saved:", imdb_small_path)

Saved: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed\imdb_small_fr.csv


In [7]:
imdb = pd.read_csv(DATA_PROCESSED / "imdb_small_fr.csv")
wanted_ids = set(imdb["tconst"])
print("wanted_ids:", len(wanted_ids))

wanted_ids: 37167


In [8]:
director_ids = set()

for chunk in pd.read_csv(
    DATA_RAW / "title.crew.tsv.gz",
    sep="\t", compression="gzip",
    usecols=["tconst","directors"],
    na_values="\\N",
    chunksize=400_000,
    low_memory=True
):
    chunk = chunk[chunk["tconst"].isin(wanted_ids)]
    for s in chunk["directors"].dropna().astype(str):
        director_ids.update(s.split(","))

print("director_ids:", len(director_ids))

director_ids: 18826


In [9]:
actor_ids = set()

for chunk in pd.read_csv(
    DATA_RAW / "title.principals.tsv.gz",
    sep="\t", compression="gzip",
    usecols=["tconst","nconst","category"],
    na_values="\\N",
    chunksize=400_000,
    low_memory=True
):
    chunk = chunk[chunk["tconst"].isin(wanted_ids)]
    chunk = chunk[chunk["category"].isin(["actor","actress"])]
    actor_ids.update(chunk["nconst"].dropna().astype(str).tolist())

print("actor_ids:", len(actor_ids))

actor_ids: 140427


In [10]:
name_file = DATA_RAW / "name.basics.tsv.gz"
if not name_file.exists():
    name_file = DATA_RAW / "name.basics .tsv.gz"

needed_nconst = director_ids.union(actor_ids)
print("needed_nconst:", len(needed_nconst))

name_map = {}

for chunk in pd.read_csv(
    name_file,
    sep="\t", compression="gzip",
    usecols=["nconst","primaryName"],
    na_values="\\N",
    chunksize=500_000,
    low_memory=True
):
    chunk = chunk[chunk["nconst"].isin(needed_nconst)]
    name_map.update(dict(zip(chunk["nconst"].astype(str), chunk["primaryName"].astype(str))))

print("name_map loaded:", len(name_map))

needed_nconst: 156571
name_map loaded: 156570


In [11]:
def first_director_name(directors):
    if pd.isna(directors):
        return None
    first_id = str(directors).split(",")[0]
    return name_map.get(first_id, first_id)

crew_rows = []

for chunk in pd.read_csv(
    DATA_RAW / "title.crew.tsv.gz",
    sep="\t", compression="gzip",
    usecols=["tconst","directors"],
    na_values="\\N",
    chunksize=400_000,
    low_memory=True
):
    chunk = chunk[chunk["tconst"].isin(wanted_ids)]
    if len(chunk) == 0:
        continue
    chunk["director_names"] = chunk["directors"].apply(first_director_name)
    crew_rows.append(chunk[["tconst","director_names"]])

crew_small = pd.concat(crew_rows, ignore_index=True)
imdb = imdb.merge(crew_small, on="tconst", how="left")

print("imdb + directors:", imdb.shape)

imdb + directors: (37167, 13)


In [12]:
# On stocke temporairement les (tconst, ordering, cast_name)
cast_rows = []

for chunk in pd.read_csv(
    DATA_RAW / "title.principals.tsv.gz",
    sep="\t", compression="gzip",
    usecols=["tconst","ordering","nconst","category"],
    na_values="\\N",
    chunksize=400_000,
    low_memory=True
):
    chunk = chunk[chunk["tconst"].isin(wanted_ids)]
    chunk = chunk[chunk["category"].isin(["actor","actress"])]
    if len(chunk) == 0:
        continue

    chunk["ordering"] = pd.to_numeric(chunk["ordering"], errors="coerce")
    chunk["cast_name"] = chunk["nconst"].astype(str).map(name_map)

    cast_rows.append(chunk[["tconst","ordering","cast_name"]])

cast = pd.concat(cast_rows, ignore_index=True)
cast = cast.dropna(subset=["cast_name"])
cast = cast.sort_values(["tconst","ordering"])

cast = cast.groupby("tconst").head(5)

casting_top = (
    cast.groupby("tconst")["cast_name"]
        .apply(lambda s: ", ".join(s.astype(str).tolist()))
        .reset_index()
        .rename(columns={"cast_name":"casting_top"}))

imdb = imdb.merge(casting_top, on="tconst", how="left")
print("imdb + casting:", imdb.shape)

imdb + casting: (37167, 14)


In [13]:
tmdb_keep = ["imdb_id","overview","tagline","poster_path","backdrop_path",
             "production_companies_name","production_countries","spoken_languages",
             "original_language","release_date","budget","revenue","genres"]

tmdb_rows = []
for chunk in pd.read_csv(DATA_RAW / "tmdb_full.csv", usecols=tmdb_keep, chunksize=200_000):
    chunk["imdb_id"] = chunk["imdb_id"].astype(str).str.strip()
    chunk = chunk[chunk["imdb_id"].isin(wanted_ids)]
    if len(chunk) == 0:
        continue
    tmdb_rows.append(chunk)

tmdb = pd.concat(tmdb_rows, ignore_index=True)

IMG = "https://image.tmdb.org/t/p/w500"
tmdb["poster_url"] = tmdb["poster_path"].apply(lambda x: IMG+str(x) if pd.notna(x) and str(x)!="nan" else None)
tmdb["backdrop_url"] = tmdb["backdrop_path"].apply(lambda x: IMG+str(x) if pd.notna(x) and str(x)!="nan" else None)

tmdb = tmdb.rename(columns={"overview":"synopsis",
    "production_companies_name":"production_companies",
    "spoken_languages":"spoken_languages_str"})

imdb = imdb.merge(tmdb, left_on="tconst", right_on="imdb_id", how="left").drop(columns=["imdb_id"], errors="ignore")

print("imdb + tmdb:", imdb.shape)

imdb + tmdb: (37167, 28)


In [14]:
import ast


def parse_genres(x):
    """Retourne une liste de genres propres depuis:
       - liste/tuple/np.array
       - string "Drama,Crime"
       - string "['Drama','Crime']"
    """
    # NaN / None
    if x is None:
        return []
    # si c'est un float nan
    if isinstance(x, float) and np.isnan(x):
        return []

    # déjà une liste/tuple/array
    if isinstance(x, (list, tuple, np.ndarray)):
        out = []
        for g in x:
            if g is None:
                continue
            if isinstance(g, float) and np.isnan(g):
                continue
            s = str(g).strip().strip("'").strip('"')
            if s and s != "\\N":
                out.append(s)
        return out

    # sinon string
    s = str(x).strip()
    if s in ["", "\\N", "nan", "None", "0"]:
        return []

    # format liste string: "['Drama', 'Crime']"
    if s.startswith("[") and s.endswith("]"):
        try:
            lst = ast.literal_eval(s)
            if isinstance(lst, list):
                out = []
                for g in lst:
                    g = str(g).strip().strip("'").strip('"')
                    if g and g != "\\N":
                        out.append(g)
                return out
        except Exception:
            pass

    # format csv: "Drama,Crime"
    parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
    return [p for p in parts if p and p != "\\N"]

def merge_genres(gx, gy, max_genres=4):
    gx_list = parse_genres(gx)
    gy_list = parse_genres(gy)

    # si identiques (même set), pas besoin d'ajouter gy
    if set(gx_list) == set(gy_list):
        merged_source = gx_list
    else:
        merged_source = gx_list + gy_list

    # dédoublonner en conservant l'ordre, limite à max_genres
    seen = set()
    merged = []
    for g in merged_source:
        if g not in seen:
            seen.add(g)
            merged.append(g)
        if len(merged) >= max_genres:
            break
    return merged

# Créer la colonne finale imdb["genres"]
imdb["genres"] = [
    merge_genres(gx, gy, max_genres=4)
    for gx, gy in zip(imdb["genres_x"].tolist(), imdb["genres_y"].tolist())
]

# Vérifs rapides
print("Exemples (genres_x / genres_y / genres):")
display(imdb[["genres_x", "genres_y", "genres"]].head(10))

all_genres = sorted({g for gs in imdb["genres"] for g in gs})
print("Nombre de genres uniques :", len(all_genres))

print("Top 30 genres fréquents :")
display(imdb["genres"].explode().value_counts().head(30))


Exemples (genres_x / genres_y / genres):


,genres_x,genres_y,genres
0,Drama,['History'],"[Drama, History]"
1,"Adventure,Drama,Fantasy","['Adventure', 'Fantasy', 'Horror']","[Adventure, Drama, Fantasy, Horror]"
2,"Biography,Drama,Romance","['Drama', 'History']","[Biography, Drama, Romance, History]"
3,"Drama,History","['Drama', 'History']","[Drama, History]"
4,"Crime,Drama",NaN,"[Crime, Drama]"
5,"Crime,Drama,Mystery",['Crime'],"[Crime, Drama, Mystery]"
6,"Drama,Fantasy,Horror","['Drama', 'Fantasy']","[Drama, Fantasy, Horror]"
7,"Adventure,Drama","['Drama', 'Action']","[Adventure, Drama, Action]"
8,"Crime,Drama,Horror","['Drama', 'Horror']","[Crime, Drama, Horror]"
9,"Drama,Western",['Western'],"[Drama, Western]"


Nombre de genres uniques : 26
Top 30 genres fréquents :


genres
Drama              23852
Comedy             10505
Romance             6987
Crime               5622
Documentary         4921
Thriller            4349
Action              4127
Adventure           3515
Mystery             2375
History             2076
Family              1954
Biography           1901
Music               1824
Fantasy             1796
Horror              1752
War                 1656
Animation           1417
Western              973
Musical              836
Science Fiction      829
Sci-Fi               745
Film-Noir            537
Sport                535
TV Movie              67
News                  31
Reality-TV             1
Name: count, dtype: int64

In [15]:
# =========================
# GENRES : fusion + nettoyage + filtre Creuse
# =========================

import ast
import numpy as np
import pandas as pd


# ---------- Parsing ----------

def parse_genres(x):

    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []

    if isinstance(x, (list, tuple, np.ndarray)):
        out = []
        for g in x:
            if g is None:
                continue
            if isinstance(g, float) and np.isnan(g):
                continue
            s = str(g).strip().strip("'").strip('"')
            if s and s != "\\N":
                out.append(s)
        return out

    s = str(x).strip()
    if s in ["", "\\N", "nan", "None", "0"]:
        return []

    # "['Drama','Crime']"
    if s.startswith("[") and s.endswith("]"):
        try:
            lst = ast.literal_eval(s)
            if isinstance(lst, list):
                return [
                    str(g).strip().strip("'").strip('"')
                    for g in lst
                    if str(g).strip() and str(g).strip() != "\\N"
                ]
        except Exception:
            pass

    # "Drama,Crime"
    return [p.strip().strip("'").strip('"') for p in s.split(",") if p.strip()]


# ---------- Fusion ----------

def merge_genres(gx, gy, max_genres=6):

    gx_list = parse_genres(gx)
    gy_list = parse_genres(gy)

    merged_source = gx_list + gy_list

    seen = set()
    merged = []

    for g in merged_source:
        if g not in seen:
            seen.add(g)
            merged.append(g)
        if len(merged) >= max_genres:
            break

    return merged


# ---------- Traduction FR ----------

GENRE_TO_FR = {
    "Drama": "Drame",
    "Comedy": "Comédie",
    "Romance": "Romance",
    "Biography": "Biographie",
    "History": "Histoire",
    "War": "Guerre",
    "Crime": "Crime",
    "Family": "Famille",
    "Documentary": "Documentaire",
    "Mystery": "Mystère",
    "Western": "Western",
    "Film-Noir": "Film noir",
    "Musical": "Comédie musicale",
    "Music": "Musique",
    "Sport": "Sport",
    "Thriller": "Thriller",
    "Action": "Action",
    "Adventure": "Aventure",
    "Fantasy": "Fantastique",
    "Animation": "Animation",
    "Horror": "Horreur",
    "Sci-Fi": "Science-fiction",
    "Science Fiction": "Science-fiction",
    "TV Movie": "Téléfilm",
    "News": "Actualités"
}


# ---------- Genres autorisés (Creuse on exclue Horreur Animation Téléfilm Actualités et Sport)

CREUSE_FORBIDDEN = {
    "Sport",
    "Horreur",
    "Téléfilm",
    "Actualités"
}


# =========================
# Création colonne genres
# =========================

if "genres_x" not in imdb.columns and "genres_y" not in imdb.columns:
    raise KeyError("Aucune colonne genres_x / genres_y dans imdb.")

gx = imdb["genres_x"] if "genres_x" in imdb.columns else [None] * len(imdb)
gy = imdb["genres_y"] if "genres_y" in imdb.columns else [None] * len(imdb)

# Fusion EN
imdb["genres_en"] = [
    merge_genres(a, b, max_genres=6)
    for a, b in zip(gx.tolist(), gy.tolist())
]

# Traduction FR
imdb["genres_fr"] = imdb["genres_en"].apply(
    lambda gs: [GENRE_TO_FR.get(g, g) for g in gs]
)

# =========================
# Filtre métier (Creuse)
# =========================

def is_allowed_movie(genres_fr):
    if not genres_fr:
        return False
    return not any(g in CREUSE_FORBIDDEN for g in genres_fr)

imdb = imdb[imdb["genres_fr"].apply(is_allowed_movie)].copy()

print("Après filtre Creuse :", imdb.shape)


# ---------- main_genre ----------

imdb["main_genre"] = imdb["genres_fr"].apply(lambda gs: gs[0] if gs else None)


# =========================
# Patrimonial
# =========================

imdb["patrimonial"] = np.where(
    (imdb["startYear"] < 1990) &
    (imdb["averageRating"] >= 8) &
    (imdb["original_language"] == "fr"),
    "Yes",
    "No"
)


# =========================
# FINAL
# =========================

cols_wanted = [
    "tconst","titleType","primaryTitle","originalTitle","title_fr",
    "genres_fr","main_genre",
    "runtimeMinutes","averageRating","numVotes","decade",
    "director_names","casting_top",
    "synopsis","tagline",
    "poster_url","backdrop_url",
    "production_companies","production_countries","spoken_languages_str",
    "original_language","release_date","budget","revenue",
    "patrimonial"]

cols_present = [c for c in cols_wanted if c in imdb.columns]

final = imdb[cols_present].copy().rename(columns={
    "tconst": "movieID",
    "genres_fr": "genres"
})

print("final:", final.shape)
final.head()


Après filtre Creuse : (34486, 31)
final: (34486, 25)


,movieID,titleType,primaryTitle,originalTitle,title_fr,genres,main_genre,runtimeMinutes,averageRating,numVotes,...,poster_url,backdrop_url,production_companies,production_countries,spoken_languages_str,original_language,release_date,budget,revenue,patrimonial
0,tt0001790,movie,"Les Misérables, Part 1: Jean Valjean",Les misérables - Époque 1: Jean Valjean,Les misérables - Époque 1: Jean Valjean,"[Drame, Histoire]",Drame,60.0,6.0,66,...,None,None,['Pathé Frères'],['FR'],['fr'],en,1913-01-01,0.0,0.0,No
2,tt0002423,movie,Passion,Madame DuBarry,Passion,"[Biographie, Drame, Romance, Histoire]",Biographie,113.0,6.7,1113,...,https://image.tmdb.org/t/p/w500/U20aqKl3pgeW3m...,https://image.tmdb.org/t/p/w500/4SadYEqr7OY8XF...,['Projektions-AG Union (PAGU)'],['DE'],['xx'],de,1919-09-18,0.0,0.0,No
3,tt0002445,movie,Quo Vadis?,Quo Vadis?,Quo Vadis?,"[Drame, Histoire]",Drame,120.0,6.1,519,...,https://image.tmdb.org/t/p/w500/f79ax2LZMU8w4s...,https://image.tmdb.org/t/p/w500/tfYNBokUsjjwzd...,['Società Italiana Cines'],['IT'],['it'],la,1913-03-01,0.0,0.0,No
4,tt0003037,movie,Fantomas: The Man in Black,Fantômas II: Juve contre Fantômas,Fantômas II: Juve contre Fantômas,"[Crime, Drame]",Crime,61.0,6.9,1836,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
5,tt0003165,movie,Fantômas: The Dead Man Who Killed,Fantômas III: Le mort qui tue,Fantômas III: Le Mort Qui Tue,"[Crime, Drame, Mystère]",Crime,90.0,6.9,1463,...,https://image.tmdb.org/t/p/w500/t9WfEu3EAhQoxM...,None,[],[],[],it,1913-01-01,0.0,0.0,No


In [16]:
nb_yes = (final["patrimonial"] == "Yes").sum()
print("Nombre de films patrimoniaux :", nb_yes)

Nombre de films patrimoniaux : 29


In [17]:
# convertir decade en numérique (sécurité)
final["decade"] = pd.to_numeric(final["decade"], errors="coerce")

# garder seulement les décennies 1950 → 2020
final = final[final["decade"].between(1950, 2020)].copy()

print("final after decade filter:", final.shape)

final after decade filter: (29412, 25)


In [18]:
missing = pd.DataFrame({
    "missing_count": final.isna().sum(),
    "missing_pct": (final.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing

,missing_count,missing_pct
tagline,19418,66.02
backdrop_url,9230,31.38
poster_url,5508,18.73
synopsis,5328,18.12
release_date,4946,16.82
revenue,4782,16.26
budget,4782,16.26
original_language,4782,16.26
spoken_languages_str,4782,16.26
production_countries,4782,16.26


In [19]:
# colonnes critiques (celles avec beaucoup de manquants chez toi)
cols_critical = [
    "backdrop_url",
    "poster_url",
    "synopsis",
    "release_date",
    "revenue",
    "budget",
    "original_language",
    "spoken_languages_str",
    "production_countries",
    "production_companies",
    "casting_top",
    "director_names"
]

# on garde seulement celles qui existent vraiment dans final
cols_critical_present = [c for c in cols_critical if c in final.columns]

print("Colonnes utilisées pour dropna:", cols_critical_present)

# suppression des lignes incomplètes uniquement sur ces colonnes
final = final.dropna(subset=cols_critical_present).copy()

print("final after selective dropna:", final.shape)
final.head(5)

Colonnes utilisées pour dropna: ['backdrop_url', 'poster_url', 'synopsis', 'release_date', 'revenue', 'budget', 'original_language', 'spoken_languages_str', 'production_countries', 'production_companies', 'casting_top', 'director_names']
final after selective dropna: (18753, 25)


,movieID,titleType,primaryTitle,originalTitle,title_fr,genres,main_genre,runtimeMinutes,averageRating,numVotes,...,poster_url,backdrop_url,production_companies,production_countries,spoken_languages_str,original_language,release_date,budget,revenue,patrimonial
3331,tt0035423,movie,Kate & Leopold,Kate & Leopold,Kate et Léopold,"[Comédie, Fantastique, Romance]",Comédie,118.0,6.4,93331,...,https://image.tmdb.org/t/p/w500/mUvikzKJJSg9kh...,https://image.tmdb.org/t/p/w500/hfeiSfWYujh6MK...,"['Konrad Pictures', 'Miramax']",['US'],"['en', 'fr', 'it']",en,2001-12-25,48000000.0,76019048.0,No
3524,tt0036606,movie,"Another Time, Another Place","Another Time, Another Place",Les Coeurs captifs,"[Drame, Guerre]",Drame,118.0,6.4,377,...,https://image.tmdb.org/t/p/w500/anoPMnxdrL4B7E...,https://image.tmdb.org/t/p/w500/i1idV6BigO38xd...,"['Umbrella', 'Associated-Rediffusion Televisio...",['GB'],"['en', 'it']",it,1983-05-13,0.0,0.0,No
4274,tt0040266,movie,The Dancing Years,The Dancing Years,Le Temps des valses,"[Drame, Comédie musicale, Romance, Musique]",Drame,98.0,6.1,85,...,https://image.tmdb.org/t/p/w500/iRAYV8zPndtMU4...,https://image.tmdb.org/t/p/w500/2PvtmXQycbJ9ff...,['Associated British Picture Corporation'],['GB'],['en'],en,1950-07-19,0.0,0.0,No
4348,tt0040559,movie,The Machine to Kill Bad People,La macchina ammazzacattivi,La machine à tuer les méchants,"[Comédie, Fantastique]",Comédie,83.0,6.7,1092,...,https://image.tmdb.org/t/p/w500/6A7qyiMkYRZNyu...,https://image.tmdb.org/t/p/w500/hBovWaO3XKUTtV...,['Tevere Film'],['IT'],['it'],it,1952-05-14,0.0,0.0,No
4474,tt0041117,movie,Ambush,Ambush,Embuscade,"[Drame, Western]",Drame,90.0,6.5,930,...,https://image.tmdb.org/t/p/w500/cmmfwAJH1AxV6m...,https://image.tmdb.org/t/p/w500/lSE957xunNt6ab...,['Metro-Goldwyn-Mayer'],['US'],['en'],en,1950-01-13,0.0,0.0,No


In [20]:
# Rename (inchangé)
final = final.rename(columns={
    "movieID": "id_film",
    "titleType": "type_titre",
    "primaryTitle": "titre_principal",
    "originalTitle": "titre_original",
    "title_fr": "titre_francais",
    "genres": "genres",
    "runtimeMinutes": "duree_minutes",
    "averageRating": "note_moyenne",
    "numVotes": "nombre_votes",
    "decade": "decennie",
    "director_names": "noms_realisateurs",
    "casting_top": "principaux_acteurs",
    "synopsis": "synopsis",
    "tagline": "slogan",
    "poster_url": "url_affiche",
    "backdrop_url": "url_fond",
    "production_companies": "societes_production",
    "production_countries": "pays_production",
    "spoken_languages_str": "langues_parlees",
    "original_language": "langue_originale",
    "release_date": "date_sortie",
    "budget": "budget",
    "revenue": "recettes",
    "patrimonial": "patrimonial"
})


# Vérif
print(final["genres"].head(10))
print(sorted({g for gs in final["genres"] for g in gs}))


3331                [Comédie, Fantastique, Romance]
3524                                [Drame, Guerre]
4274    [Drame, Comédie musicale, Romance, Musique]
4348                         [Comédie, Fantastique]
4474                               [Drame, Western]
4478                               [Drame, Romance]
4481                      [Drame, Comédie musicale]
4493            [Crime, Film noir, Thriller, Drame]
4503                    [Action, Aventure, Romance]
4513                                        [Drame]
Name: genres, dtype: object
['Action', 'Animation', 'Aventure', 'Biographie', 'Comédie', 'Comédie musicale', 'Crime', 'Documentaire', 'Drame', 'Famille', 'Fantastique', 'Film noir', 'Guerre', 'Histoire', 'Musique', 'Mystère', 'Romance', 'Science-fiction', 'Thriller', 'Western']


In [21]:
vals = (
    final["pays_production"]
    .dropna()
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)


vals = vals[vals.ne("")].unique()
vals
import re
import pandas as pd


def clean_country_codes(x):
    if pd.isna(x):
        return []
    s = str(x).strip()


    # Normalise les séparateurs (au cas où il y ait ; | /)
    s = re.sub(r"[;|/]", ",", s)


    # Enlève crochets et guillemets (si jamais tu as encore des formats type "['US','FR']")
    s = re.sub(r"[\[\]\"']", "", s)


    # Split + strip
    parts = [p.strip() for p in s.split(",")]


    # Filtre les vides
    return [p for p in parts if p]


countries_clean = (
    final["pays_production"]
    .apply(clean_country_codes)
    .explode()
)



countries_clean.unique()


EUROPE = {
    "AL","AD","AT","BA","BE","BG","BY","CH","CY","CZ","DE","DK","EE","ES","FI","FR","GB","GE",
    "GR","HR","HU","IE","IS","IT","LI","LT","LU","LV","MC","MD","ME","MK","MT","NL","NO","PL",
    "PT","RO","RS","RU","SE","SI","SK","SM","TR","UA","VA","XK",
    # anciens codes possibles
    "SU","YU","CS"
}


def zone_de_production(x):
    codes = set(clean_country_codes(x))


    if "FR" in codes:
        return "Français"
    if "US" in codes:
        return "Américain"
    if codes & EUROPE:
        return "Européen"
    return "Autres"


final["zone_de_production"] = final["pays_production"].apply(zone_de_production)

final.shape

(18753, 26)

In [34]:
# Dictionnaire des seuils
seuils_votes = {
    "Français": 100,
    "Américain": 5000,
    "Européen": 2000,
    "Autres": 10000
}
final =final[
    final["nombre_votes"] >= final["zone_de_production"].map(seuils_votes)
]

print(final.shape)

(10526, 26)


In [35]:
# Séparer les films "Autres"
mask_autres = final["zone_de_production"] == "Autres"

autres = final[mask_autres]
reste = final[~mask_autres]

# Seuil : top 50% sur la note moyenne
threshold = autres["note_moyenne"].quantile(0.50)

print("Seuil top 50% (Autres) :", threshold)

# Garder seulement les meilleurs "Autres"
autres_top = autres[autres["note_moyenne"] >= threshold]

# Recombiner
final = pd.concat([reste, autres_top]).reset_index(drop=True)

print("Après filtre Autres :", final.shape)


Seuil top 50% (Autres) : 7.5
Après filtre Autres : (10307, 26)


In [36]:
final["zone_de_production"].value_counts()


zone_de_production
Américain    4137
Français     3949
Européen     1993
Autres        228
Name: count, dtype: int64

In [38]:
import ast


iso2fr = {
    'ab': 'abkhaze',
    'aa': 'afar',
    'af': 'afrikaans',
    'ak': 'akan',
    'sq': 'albanais',
    'am': 'amharique',
    'ar': 'arabe',
    'an': 'aragonais',
    'hy': 'arménien',
    'as': 'assamais',
    'av': 'avar',
    'ae': 'avestique',
    'ay': 'aymara',
    'az': 'azéri',
    'bm': 'bambara',
    'ba': 'bachkir',
    'eu': 'basque',
    'be': 'biélorusse',
    'bn': 'bengali',
    'bh': 'bihari',
    'bi': 'bislama',
    'bs': 'bosniaque',
    'br': 'breton',
    'bg': 'bulgare',
    'my': 'birman',
    'ca': 'catalan',
    'ch': 'chamorro',
    'ce': 'tchétchène',
    'ny': 'chewa',
    'zh': 'chinois',
    'cv': 'tchouvache',
    'kw': 'cornique',
    'co': 'corse',
    'cr': 'cree',
    'hr': 'croate',
    'cs': 'tchèque',
    'da': 'danois',
    'dv': 'maldivien',
    'nl': 'néerlandais',
    'dz': 'dzongkha',
    'en': 'anglais',
    'eo': 'espéranto',
    'et': 'estonien',
    'ee': 'éwé',
    'fo': 'féroïen',
    'fj': 'fidjien',
    'fi': 'finnois',
    'fr': 'français',
    'ff': 'peul',
    'gd': 'gaélique écossais',
    'gl': 'galicien',
    'lg': 'ganda',
    'ka': 'géorgien',
    'de': 'allemand',
    'el': 'grec',
    'kl': 'groenlandais',
    'gn': 'guarani',
    'gu': 'goudjrati',
    'ht': 'haïtien',
    'ha': 'haoussa',
    'he': 'hébreu',
    'hz': 'herero',
    'hi': 'hindi',
    'ho': 'hiri motu',
    'hu': 'hongrois',
    'ia': 'interlingua',
    'id': 'indonésien',
    'ie': 'interlingue',
    'ga': 'irlandais',
    'ig': 'igbo',
    'ik': 'inupiaq',
    'io': 'ido',
    'is': 'islandais',
    'it': 'italien',
    'iu': 'inuktitut',
    'ja': 'japonais',
    'jv': 'javanais',
    'kl': 'groenlandais',
    'kn': 'kannada',
    'kr': 'kanouri',
    'ks': 'kashmiri',
    'kk': 'kazakh',
    'km': 'khmer',
    'ki': 'kikuyu',
    'rw': 'kinyarwanda',
    'ky': 'kirghiz',
    'kv': 'komi',
    'kg': 'kongo',
    'ko': 'coréen',
    'ku': 'kurde',
    'kj': 'kwanyama',
    'la': 'latin',
    'lb': 'luxembourgeois',
    'lg': 'ganda',
    'li': 'limbourgeois',
    'ln': 'lingala',
    'lo': 'lao',
    'lt': 'lituanien',
    'lu': 'luba-katanga',
    'lv': 'letton',
    'gv': 'manx',
    'mk': 'macédonien',
    'mg': 'malgache',
    'ms': 'malais',
    'ml': 'malayalam',
    'mt': 'maltais',
    'mi': 'maori',
    'mr': 'marathe',
    'mh': 'marshallais',
    'mn': 'mongol',
    'na': 'nauru',
    'nv': 'navajo',
    'nd': 'ndébélé du nord',
    'nr': 'ndébélé du sud',
    'ng': 'ndonga',
    'ne': 'népalais',
    'no': 'norvégien',
    'nb': 'norvégien bokmål',
    'nn': 'norvégien nynorsk',
    'ii': 'yi de Sichuan',
    'oc': 'occitan',
    'oj': 'ojibwa',
    'or': 'oriya',
    'om': 'oromo',
    'os': 'ossète',
    'pa': 'pendjabi',
    'pi': 'pali',
    'fa': 'persan',
    'pl': 'polonais',
    'pt': 'portugais',
    'ps': 'pachto',
    'qu': 'quechua',
    'rm': 'romanche',
    'rn': 'rundi',
    'ro': 'roumain',
    'ru': 'russe',
    'sa': 'sanskrit',
    'sc': 'sarde',
    'sd': 'sindhi',
    'se': 'sami du nord',
    'sm': 'samoan',
    'sg': 'sango',
    'sr': 'serbe',
    'sn': 'shona',
    'si': 'singhalais',
    'sk': 'slovaque',
    'sl': 'slovène',
    'so': 'somali',
    'st': 'sotho du sud',
    'es': 'espagnol',
    'su': 'soundanais',
    'sw': 'swahili',
    'ss': 'swati',
    'sv': 'suédois',
    'ta': 'tamoul',
    'te': 'télougou',
    'tg': 'tadjik',
    'th': 'thaï',
    'ti': 'tigrigna',
    'bo': 'tibétain',
    'tk': 'turkmène',
    'tl': 'tagalog',
    'tn': 'tswana',
    'to': 'tonga',
    'tr': 'turc',
    'ts': 'tsonga',
    'tt': 'tatar',
    'tw': 'twi',
    'ty': 'tahitien',
    'ug': 'ouïghour',
    'uk': 'ukrainien',
    'ur': 'ourdou',
    'uz': 'ouzbek',
    've': 'venda',
    'vi': 'vietnamien',
    'vo': 'volapük',
    'wa': 'wallon',
    'cy': 'gallois',
    'wo': 'wolof',
    'xh': 'xhosa',
    'yi': 'yiddish',
    'yo': 'yoruba',
    'za': 'zhuang',
    'zu': 'zoulou',
    'xx': 'inconnu'
}




def ensure_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [x]  # si ce n'est pas une vraie liste, on met dans une liste
    elif isinstance(x, list):
        return x
    else:
        return [x]  # tout le reste devient liste


# Transformer en liste
final['langues_parlees'] = final['langues_parlees'].apply(ensure_list)
final['langue_originale'] = final['langue_originale'].apply(ensure_list)


# Traduction directement dans les colonnes existantes
final['langues_parlees'] = final['langues_parlees'].apply(
    lambda lst: [iso2fr.get(code, code) for code in lst]
)


final['langue_originale'] = final['langue_originale'].apply(
    lambda lst: [iso2fr.get(code, code) for code in lst]
)


In [39]:
final["decennie"]=final["decennie"].astype(int)

In [40]:
final["date_sortie"] = pd.to_datetime(final["date_sortie"], errors="coerce")
final["annee_de_sortie"] = final["date_sortie"].dt.year


In [41]:
final["annee_de_sortie"] = final["date_sortie"].dt.year.astype("Int64")


In [42]:
final = final.dropna(subset=["annee_de_sortie"])


In [43]:
final.shape

(10307, 27)

In [44]:
sorted({g for gs in final["genres"] for g in gs})


['Action',
 'Animation',
 'Aventure',
 'Biographie',
 'Comédie',
 'Comédie musicale',
 'Crime',
 'Documentaire',
 'Drame',
 'Famille',
 'Fantastique',
 'Film noir',
 'Guerre',
 'Histoire',
 'Musique',
 'Mystère',
 'Romance',
 'Science-fiction',
 'Thriller',
 'Western']

In [45]:

#  Supprimer les films sans genre restant
final = final[final["genres"].apply(len) > 0].copy()

print("Dataset après filtre strict :", final.shape)

# Vérifier les genres restants
sorted({g for gs in final["genres"] for g in gs})

Dataset après filtre strict : (10307, 27)


['Action',
 'Animation',
 'Aventure',
 'Biographie',
 'Comédie',
 'Comédie musicale',
 'Crime',
 'Documentaire',
 'Drame',
 'Famille',
 'Fantastique',
 'Film noir',
 'Guerre',
 'Histoire',
 'Musique',
 'Mystère',
 'Romance',
 'Science-fiction',
 'Thriller',
 'Western']

In [46]:
final.shape

(10307, 27)

In [47]:
final.columns

Index(['id_film', 'type_titre', 'titre_principal', 'titre_original',
       'titre_francais', 'genres', 'main_genre', 'duree_minutes',
       'note_moyenne', 'nombre_votes', 'decennie', 'noms_realisateurs',
       'principaux_acteurs', 'synopsis', 'slogan', 'url_affiche', 'url_fond',
       'societes_production', 'pays_production', 'langues_parlees',
       'langue_originale', 'date_sortie', 'budget', 'recettes', 'patrimonial',
       'zone_de_production', 'annee_de_sortie'],
      dtype='object')

In [48]:
def to_list(x):
    # x peut être déjà une liste, ou une string type "['Drame', 'Famille']"
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "":
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else []
    except Exception:
        # fallback si jamais c'est déjà "Drame, Famille"
        return [p.strip() for p in s.split(",") if p.strip()]

# 1) liste propre
final["genres_list"] = final["genres"].apply(to_list)


In [49]:
final.columns

Index(['id_film', 'type_titre', 'titre_principal', 'titre_original',
       'titre_francais', 'genres', 'main_genre', 'duree_minutes',
       'note_moyenne', 'nombre_votes', 'decennie', 'noms_realisateurs',
       'principaux_acteurs', 'synopsis', 'slogan', 'url_affiche', 'url_fond',
       'societes_production', 'pays_production', 'langues_parlees',
       'langue_originale', 'date_sortie', 'budget', 'recettes', 'patrimonial',
       'zone_de_production', 'annee_de_sortie', 'genres_list'],
      dtype='object')

In [50]:
out = DATA_PROCESSED / "movies_fr_final.csv"
final.to_csv(out, index=False)
print("Saved:", out)

Saved: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed\movies_fr_final.csv


In [51]:
out_parquet = DATA_PROCESSED / "movies_fr_final.parquet"

final.to_parquet(out_parquet, index=False, engine="pyarrow")
print("Saved Parquet:", out_parquet)

Saved Parquet: c:\Users\azerty\Desktop\Projet2 - Copie\data\processed\movies_fr_final.parquet


In [52]:
nb_yes = (final["patrimonial"] == "Yes").sum()
print("Nombre de films patrimoniaux :", nb_yes)

Nombre de films patrimoniaux : 14


In [53]:
import pandas as pd

movies_fr_final = pd.read_csv("../data/processed/movies_fr_final.csv")

movies_fr_final.isna().sum()

id_film                   0
type_titre                0
titre_principal           0
titre_original            0
titre_francais            0
genres                    0
main_genre                0
duree_minutes             0
note_moyenne              0
nombre_votes              0
decennie                  0
noms_realisateurs         0
principaux_acteurs        0
synopsis                  0
slogan                 4463
url_affiche               0
url_fond                  0
societes_production       0
pays_production           0
langues_parlees           0
langue_originale          0
date_sortie               0
budget                    0
recettes                  0
patrimonial               0
zone_de_production        0
annee_de_sortie           0
genres_list               0
dtype: int64

In [54]:
movies_fr_final.shape 

(10307, 28)

In [55]:
movies_fr_final.info

<bound method DataFrame.info of          id_film type_titre             titre_principal  \
0      tt0035423      movie              Kate & Leopold   
1      tt0041640      movie            La Marie du Port   
2      tt0041719      movie                     Orpheus   
3      tt0041931      movie                   Stromboli   
4      tt0042003      movie     A Man Walks in the City   
...          ...        ...                         ...   
10302  tt9263550      movie  Rocketry: The Nambi Effect   
10303  tt9426210      movie         Weathering with You   
10304  tt9477520      movie                      Asuran   
10305  tt9851854      movie                       Major   
10306  tt9900782      movie                      Kaithi   

                      titre_original                 titre_francais  \
0                     Kate & Leopold                Kate et Léopold   
1                   La Marie du Port               La Marie du port   
2                             Orphée          

In [56]:
movies_fr_final.dtypes

id_film                 object
type_titre              object
titre_principal         object
titre_original          object
titre_francais          object
genres                  object
main_genre              object
duree_minutes          float64
note_moyenne           float64
nombre_votes             int64
decennie                 int64
noms_realisateurs       object
principaux_acteurs      object
synopsis                object
slogan                  object
url_affiche             object
url_fond                object
societes_production     object
pays_production         object
langues_parlees         object
langue_originale        object
date_sortie             object
budget                 float64
recettes               float64
patrimonial             object
zone_de_production      object
annee_de_sortie          int64
genres_list             object
dtype: object